# 02 · Annual indices: distributions, data quality, regions and trends

`annual_indices` is the table the challenge targets come from. The four scored metrics
(`proto_star.climate.METRICS`) map to raw columns:

| metric | column | eligible when |
|---|---|---|
| `annual_avg_temp` | `Avg Temp` (°F) | `Miss Tmax` and `Miss Tmin` ≤ 10 |
| `total_precip_in` | `ToT Prcp` (in) | `Miss Prcp` ≤ 10 |
| `hottest_day_temp` | `Max Tmax` (°F) | `Miss Tmax` ≤ 10 |
| `cdd` | `cdd` (°F-days, base 65°F) | `Miss Tmax` and `Miss Tmin` ≤ 10 |

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from proto_star.climate import MAX_MISSING_DAYS, METRICS, annual_to_long
from proto_star.data.protostar import (
    REGIONS,
    city_table,
    load_annual,
    load_city_info,
    load_coverage,
    load_daily,
)

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
REGION_PALETTE = dict(zip(REGIONS, sns.color_palette("tab10", len(REGIONS)), strict=True))

In [ ]:
annual = load_annual()
cities = city_table(annual, load_city_info())
annual = annual.merge(cities[["ID", "region"]], on="ID", how="left")
MISS = ["Miss Tmax", "Miss Tmin", "Miss Prcp"]
VALUES = [
    c for c in annual.columns if c not in {"Name", "ID", "Lat", "Lon", "Year", "region", *MISS}
]
print(len(VALUES), "value columns:", VALUES)

## 1. Missing days and the ≤10 eligibility rule

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for col in MISS:
    sns.histplot(
        annual[col], bins=np.arange(0, 370, 5), ax=axes[0], label=col, element="step", fill=False
    )
axes[0].set_yscale("log")
axes[0].axvline(MAX_MISSING_DAYS, color="k", ls="--")
axes[0].set_title("Missing days per city-year (log count)")
axes[0].legend()

eligible = annual[MISS].le(MAX_MISSING_DAYS).assign(Year=annual["Year"])
eligible.groupby((eligible["Year"] // 10) * 10)[MISS].mean().plot(ax=axes[1], marker="o")
axes[1].set_title(f"Share of city-years with ≤{MAX_MISSING_DAYS} missing days, by decade")
axes[1].set_ylim(0, 1.02)
plt.tight_layout()

In [ ]:
# City x year eligibility heatmap (temperature), cities grouped by region
ok = annual.assign(
    ok=annual[["Miss Tmax", "Miss Tmin"]].max(axis=1).le(MAX_MISSING_DAYS).astype(float)
)
grid = ok.pivot_table(index=["region", "Name"], columns="Year", values="ok", observed=True)
fig, ax = plt.subplots(figsize=(14, 9))
sns.heatmap(
    grid,
    cmap="RdYlGn",
    vmin=0,
    vmax=1,
    cbar_kws={"label": "eligible (1) / not (0)"},
    yticklabels=False,
    ax=ax,
)
ax.set_title("Temperature eligibility per city-year (white = no row); rows sorted by region")
plt.tight_layout()

In [ ]:
recent = annual[annual["Year"].between(1989, 2018)]
counts = pd.DataFrame(
    {
        "temp eligible years (of 30)": recent.assign(
            e=recent[["Miss Tmax", "Miss Tmin"]].max(axis=1).le(MAX_MISSING_DAYS)
        )
        .groupby("ID")["e"]
        .sum(),
        "precip eligible years (of 30)": recent.assign(e=recent["Miss Prcp"].le(MAX_MISSING_DAYS))
        .groupby("ID")["e"]
        .sum(),
    }
)
print("Eligible years per city in the 30-year baseline window 1989-2018:")
counts.describe().round(1)

## 2. Sentinel values and fake zeros

Years where a variable is missing for (almost) the whole year produce artefacts instead of NaN.

In [ ]:
num = annual[VALUES]
artefacts = pd.DataFrame(
    {
        "+inf": np.isposinf(num).sum(),
        "-inf": np.isneginf(num).sum(),
        "NaN": num.isna().sum(),
    }
)
artefacts[artefacts.sum(axis=1) > 0]

In [ ]:
all_missing_prcp = annual["Miss Prcp"] >= 365
print("years with ≥365 missing precip days:", int(all_missing_prcp.sum()))
print(
    "  ToT Prcp value in those years:",
    annual.loc[all_missing_prcp, "ToT Prcp"].describe()[["mean", "max"]].to_dict(),
)
print(
    "  Wet Days value in those years:",
    annual.loc[all_missing_prcp, "Wet Days"].describe()[["mean", "max"]].to_dict(),
)

partial = annual["Miss Prcp"].between(11, 364)
fig, ax = plt.subplots(figsize=(8, 4))
sns.scatterplot(data=annual[partial], x="Miss Prcp", y="ToT Prcp", s=8, alpha=0.4, ax=ax)
ax.set_title("Partially missing years: total precip is biased low as missing days grow")
plt.tight_layout()

## 3. Distributions of every index (quality-screened)

For plotting only: ±inf → NaN, and city-years with >10 missing days in **any** variable dropped.

In [ ]:
screened = annual[annual[MISS].max(axis=1).le(MAX_MISSING_DAYS)].copy()
screened[VALUES] = screened[VALUES].replace([np.inf, -np.inf], np.nan)
print(f"screened rows: {len(screened):,} of {len(annual):,}")

fig, axes = plt.subplots(5, 4, figsize=(16, 16))
for ax, col in zip(axes.flat, VALUES, strict=False):
    sns.histplot(screened[col], bins=50, ax=ax)
    ax.set_title(f"{col}  (skew {screened[col].skew():.2f})")
    ax.set_xlabel("")
for ax in axes.flat[len(VALUES) :]:
    ax.set_visible(False)
plt.tight_layout()

In [ ]:
screened[VALUES].describe(percentiles=[0.01, 0.5, 0.99]).T.round(2)

## 4. Targets by region

In [ ]:
history = annual_to_long(load_annual())  # official quality screen, long format
history = history.merge(cities[["ID", "region", "Name"]], left_on="city_id", right_on="ID")
history.groupby("metric")["actual_value"].describe().round(2)

In [ ]:
last30 = history[history["year"].between(1989, 2018)]
fig, axes = plt.subplots(2, 2, figsize=(15, 9))
for ax, metric in zip(axes.flat, METRICS, strict=True):
    sns.boxplot(
        data=last30[last30["metric"] == metric],
        x="region",
        y="actual_value",
        hue="region",
        palette=REGION_PALETTE,
        ax=ax,
        fliersize=1,
        legend=False,
    )
    ax.set_title(f"{metric}, 1989-2018")
    ax.tick_params(axis="x", rotation=20)
plt.tight_layout()

In [ ]:
# Regional median per year + 10-year rolling mean
regional = (
    history.groupby(["metric", "region", "year"], observed=True)["actual_value"]
    .median()
    .reset_index()
)
regional = regional[regional["year"] >= 1900]
fig, axes = plt.subplots(2, 2, figsize=(16, 9), sharex=True)
for ax, metric in zip(axes.flat, METRICS, strict=True):
    for region, group in regional[regional["metric"] == metric].groupby("region", observed=True):
        series = group.set_index("year")["actual_value"]
        ax.plot(series.index, series, color=REGION_PALETTE[region], alpha=0.25, lw=0.8)
        ax.plot(
            series.index,
            series.rolling(10, min_periods=5).mean(),
            color=REGION_PALETTE[region],
            label=region,
        )
    ax.set_title(f"{metric}: regional median (thin) and 10y rolling mean")
axes.flat[0].legend(fontsize=8)
plt.tight_layout()

## 5. Scale and variability per city

The official score is MAE skill relative to a 30-year mean, pooled over all cities per metric.
Cities with large year-to-year swings dominate a pooled MAE, so the spread across cities matters.

In [ ]:
stats = (
    last30.groupby(["metric", "city_id"])["actual_value"]
    .agg(["mean", "std", "count"])
    .reset_index()
)
stats = stats.merge(cities[["ID", "region"]], left_on="city_id", right_on="ID")
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, metric in zip(axes, METRICS, strict=True):
    sns.scatterplot(
        data=stats[stats["metric"] == metric],
        x="mean",
        y="std",
        hue="region",
        palette=REGION_PALETTE,
        s=18,
        ax=ax,
        legend=ax is axes[-1],
    )
    ax.set_title(f"{metric}: per-city mean vs std (1989-2018)")
axes[-1].legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8)
plt.tight_layout()
stats.groupby("metric")["std"].describe().round(2)

## 6. Recent trends (1989-2018 OLS slope per city)

In [ ]:
def slope_per_decade(group: pd.DataFrame) -> float:
    if len(group) < 20:
        return np.nan
    return float(np.polyfit(group["year"], group["actual_value"], 1)[0] * 10)


slopes = (
    last30.groupby(["metric", "city_id"])[["year", "actual_value"]]
    .apply(slope_per_decade)
    .rename("slope_per_decade")
    .reset_index()
    .merge(cities[["ID", "Lat", "Lon", "region"]], left_on="city_id", right_on="ID")
)
slopes.groupby("metric")["slope_per_decade"].describe().round(3)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 9))
for ax, metric in zip(axes.flat, METRICS, strict=True):
    part = slopes[slopes["metric"] == metric].dropna()
    lim = part["slope_per_decade"].abs().quantile(0.95)
    points = ax.scatter(
        part["Lon"],
        part["Lat"],
        c=part["slope_per_decade"],
        cmap="coolwarm",
        vmin=-lim,
        vmax=lim,
        s=30,
        edgecolor="k",
        linewidth=0.3,
    )
    plt.colorbar(points, ax=ax, label="per decade")
    ax.set_title(f"{metric}: 1989-2018 trend per decade (cities with ≥20 eligible years)")
plt.tight_layout()

## 7. Correlation between indices (screened city-years)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(
    screened[VALUES].corr(),
    cmap="vlag",
    center=0,
    annot=True,
    fmt=".2f",
    annot_kws={"size": 7},
    ax=ax,
)
ax.set_title("Pearson correlation, pooled over cities and years")
plt.tight_layout()

In [ ]:
# Pooled correlations mix geography with year-to-year variation. Remove each city's mean
# to see how indices co-vary *within* a city (what matters for forecasting anomalies).
within = screened[VALUES] - screened.groupby("ID")[VALUES].transform("mean")
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(
    within.corr(), cmap="vlag", center=0, annot=True, fmt=".2f", annot_kws={"size": 7}, ax=ax
)
ax.set_title("Correlation of within-city anomalies")
plt.tight_layout()

## 8. Station changes and the target series

For every station switch (`Station Changes > 0` in `coverage_and_stations`), compare the mean of
each metric's **region-relative anomaly** over up to 5 eligible years after the switch with the
5 years before it. Subtracting the regional median for each year removes shared weather.
**Placebo:** the same before/after difference computed at years with no switch. If switches
matter, their step distribution should be wider than the placebo's.

In [ ]:
coverage = load_coverage()
hist = history.copy()
hist["anom"] = hist["actual_value"] - hist.groupby(["metric", "region", "year"], observed=True)[
    "actual_value"
].transform("median")
switch_years = coverage.loc[coverage["Station Changes"] > 0, ["ID", "Year"]]
placebo_years = coverage.loc[coverage["Station Changes"] == 0, ["ID", "Year"]].sample(
    len(switch_years) * 3, random_state=0
)


def step_sizes(events: pd.DataFrame, window: int = 5) -> pd.DataFrame:
    rows = []
    series = {key: g.set_index("year")["anom"] for key, g in hist.groupby(["city_id", "metric"])}
    for city_id, year in events.itertuples(index=False):
        for metric in METRICS:
            s = series.get((city_id, metric))
            if s is None:
                continue
            before = s[(s.index < year) & (s.index >= year - window)]
            after = s[(s.index > year) & (s.index <= year + window)]
            if len(before) >= 3 and len(after) >= 3:
                rows.append({"metric": metric, "step": after.mean() - before.mean()})
    return pd.DataFrame(rows)


steps = pd.concat(
    [
        step_sizes(switch_years).assign(kind="station switch"),
        step_sizes(placebo_years).assign(kind="placebo (no switch)"),
    ]
)
steps.groupby(["metric", "kind"])["step"].agg(
    n="size", mean="mean", std="std", abs_median=lambda s: s.abs().median()
).round(2)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, metric in zip(axes, METRICS, strict=True):
    sns.histplot(
        data=steps[steps["metric"] == metric],
        x="step",
        hue="kind",
        stat="density",
        common_norm=False,
        element="step",
        bins=30,
        ax=ax,
        legend=ax is axes[-1],
    )
    ax.axvline(0, color="k", lw=0.8)
    ax.set_title(f"{metric}: 5y after - 5y before")
plt.tight_layout()

## Findings: annual indices and preprocessing implications

**Data quality**
- **Years that are entirely missing produce sentinel values, not NaN.** When a whole year is
  missing, `Max`/`Min` columns hold ±inf (1,174 to 1,184 rows) and `Max 1-day P` holds -inf
  (932 rows). **`ToT Prcp` and `Wet Days` are exactly 0** in all 932 years with no precipitation
  data, which looks like a real (fake) zero. Precipitation totals in partially missing years are
  also biased low. The official screen (`annual_to_long`, ≤10 missing days) removes all of these,
  but any feature built from the raw columns must apply the same screen or convert them to NaN.
- About 89% of city-years pass the ≤10 rule. Most failures are before about 1900, plus a few
  long dropouts.
- **Thin 30-year baselines:** most cities have 30 of 30 eligible years in 1989-2018, but 23 have
  fewer than 25 for at least one variable. Huntsville has 1, Elko and Grand Island have 1 for
  precipitation, and Belvidere has 4 for temperature. The 30-year-mean reference and
  `ten_year_mean` behave very differently for these cities.

**Distributions and scale**
- Most indices are close to symmetric when pooled (|skew| < 0.5, and annual `ToT Prcp` is only 0.26).
  The right-skewed ones are `cdd` (1.15, with exact zeros and a range of 0 to 5,500), `Max 1-day P`
  (1.98), `Max 5-day P` (1.72) and `Wet Days` (0.78). Consider log or sqrt transforms for these.
- Between-city differences dwarf within-city variation. For example, the spread of
  `annual_avg_temp` across cities is about 9 °F, while the typical year-to-year standard deviation
  within a city is about 1.4 °F. **Model anomalies relative to a per-city climatology**
  (computed on training years only), not raw levels.
- Year-to-year noise differs by city. Precipitation standard deviation ranges from 1.6 to 16 in,
  and `cdd` from 4 to 330. The pooled MAE in the official score is therefore dominated by wet and
  hot cities.

**Regions and trends (1989-2018)**
- Warming is widespread: the median `annual_avg_temp` trend is about +0.54 °F per decade and the
  median `cdd` trend is about +90 per decade. `hottest_day_temp` and precipitation trends are
  weak and noisy, with a sign that varies by city, so trend extrapolation is risky for those.
- The level and variability of every metric depend strongly on region. Region (or lat/lon) is a
  natural grouping for pooling, but it should not replace per-city levels.
- Pooled correlations largely reflect geography. Correlations within a city show which indices
  co-vary year to year, for example Avg Temp with cdd (r = 0.70) and Warm Days with Avg Tmax
  (r = 0.65). Total precipitation is only weakly related to temperature (r = -0.14 with Avg Tmax).

**Station changes (from `coverage_and_stations`)**
- Station switches shift the temperature-based targets. The median step size for
  `annual_avg_temp` is 0.82 °F after a switch versus 0.44 °F at placebo years. The mean step is
  -0.53 °F, so the new station tends to read cooler (often a move to an airport). `cdd` shows the
  same pattern: a median of 117 versus 61, and a mean of -53. `hottest_day_temp` and precipitation
  show no clear effect.
- **Implication:** for the 30 cities with a switch in 1989-2018, a 30-year mean mixes two
  stations. Consider computing per-city levels only from the current `Station Segment`, or adding
  segment-aware features. Validate this on the backtest split before adopting it.